# Polytope Lib Tests

In [ ]:
from pycvxset import Polytope
import numpy as np
import matplotlib.pyplot as plt

import os
import mosek

os.environ["MOSEKLM_LICENSE_FILE"] = "27007@192.168.100.118"
mosek.Env().getversion()
mosek.Env().putlicensepath("27007@192.168.100.118")
mosek.Env().expirylicenses()

In [ ]:
from itertools import product
def cartesian_product(*polytopes):
    V_rep = [p.V for p in polytopes]
    Combinations = []
    for veritces in product(*V_rep):
        Combinations.append(np.concatenate(veritces))
    return Polytope(V=np.array(Combinations))

In [ ]:
# Define the polytope using H-representation with 2D

A2 = np.array([[1, 0], [0, 1], [-1, 0], [0, -1]])
b2 = np.array([1, 2, 1, 2])

P_2D = Polytope(A=A2, b=b2)

# Define the polytope using H-representation with 1D

P_1D = Polytope(A=np.array([[1], [-1]]), b=np.array([3, 3]))

# Define the polytope using H-representation with 3D as Chartesian product of 1D and 2D

P = cartesian_product(P_1D, P_2D)


ax, _, _ = P.plot()
plt.show() 

In [ ]:
F = Polytope(A = P.A/P.b[:,np.newaxis], b= np.ones((len(P.b),1)))
#ax, _, _ = F.plot()
#plt.show

P.A[:,:1],P.A[:,1:]

In [ ]:
# Check proportional scalebility of sets

sigP = 0.5 * P

ax, _, _ = sigP.plot()
plt.show() 


# Casadi Tests

In [ ]:
import numpy as np
import casadi as ca 
from pycvxset import Polytope

In [ ]:
N = 30
n = 3
m = 2
op = ca.Opti()

x_opti =op.variable(n)
op.set_initial(x_opti,[1,2,3])
print(x_opti)

Q = ca.MX(np.array([[1,0,0],[0,2,0],[0,0,3]]))

# A = np.random(n,n)
# B = np.random(n.m)

x = ca.MX.sym('x',n,1,N)
x[0]=ca.MX(np.array([1,2,3]))
u = ca.MX.sym('u',m,1,N)
print(Q)
print(Q@x[0])
# constraint = []

# for i in range(N):

#     constraint.append(x[i+1]==A @ x[i] + B @ u[i])
#     ca.qpsol()



## Casadi - Solving a NLP with Opti

In [ ]:
opti = ca.Opti()

x = opti.variable()
y = opti.variable()

opti.minimize((y-x**2)**2)
opti.subject_to(x**2+y**2==1)
opti.subject_to(x+y>1)

opti.solver('ipopt')

sol = opti.solve()

print(sol.value(x))
print(sol.value(y))

## Casadi - Solving a NLP with MX

In [ ]:
x = ca.MX.sym('x')
y = ca.MX.sym('y')
z = ca.MX.sym('z')

F = ca.MX.sym("F")
G = ca.MX.sym("G")
f = (y-x**2)**2
g = ca.vertcat(x**2+y**2,x+y)
nlp = {'x':ca.vertcat(x,y), 'f':f, 'g':g, 'p':z}
s = ca.nlpsol('s', 'ipopt', nlp)
r = s(lbg=ca.vertcat(1,1+z), ubg=ca.vertcat(1,np.abs(-np.inf)),x0=0.786151)
x_opt = r['x']
print(x_opt)


In [ ]:
z = ca.vertcat(x,y)
Const = ca.horzcat(F,G)
Const@z
g = ca.vertcat(g,Const@z)
const_value = []
h = np.array([[1,1,1],[2,2,2],[3,3,3]])
H = ca.MX(h)
print(H)

## Transforming a MX object into a numpy array - need to pass by DM object

In [ ]:
print(H)

g=ca.MX()

g=ca.vertcat(g,H)
g

# CVXPY LMI Solver

In [ ]:
import cvxpy as cp
import numpy as np

import os
import mosek

os.environ["MOSEKLM_LICENSE_FILE"] = "27007@192.168.100.118"
mosek.Env().getversion()
mosek.Env().putlicensepath("27007@192.168.100.118")
mosek.Env().expirylicenses()

# Data
A = np.array([[0, 1], [-1, -2]])
n = 2

# Variables
P = cp.Variable((n, n), symmetric=True)

tol = cp.Parameter(1,"tol")

# Constraints
# P > 0 and A.T*P + P*A < 0 (using a small epsilon for strict inequality)
constraints = [P >> tol * np.eye(n), 
               A.T @ P + P @ A << -tol * np.eye(n)]

# Problem
prob = cp.Problem(cp.Minimize(0), constraints)
#prob.solve()

#print("Status:", prob.status)
#print("P:\n", P.value)


In [ ]:
tol.value = [1e-6]
prob.solve()
print("Status:", prob.status)
print("P:\n", P.value)

In [ ]:
import scipy as sc
l=cp.Parameter(1,"lambda")
print(l)
l.shape

A = np.array([[1,2],[2,1]]).transpose()
print(np.linalg.inv(A))

In [ ]:
rho = np.array([[.9, .1, 2]])

if np.isnan(rho).any():
    print("Hello")

np.isnan(rho).any()

In [ ]:
import numpy as np
A=np.array([[[1, 2],[3, 4]],
            [[5, 6],[7, 8]],
            [[9, 10],[11, 12]]])
len(A.shape)
A_trans = np.copy(A).T

B=np.array([[1, 2],[3, 4]])
len(B.shape)

if len(A.shape)>2:
    v=A.shape
else:
    v=1


if len(B.shape)==2:
    B = B[np.newaxis, :, :]

b = np.array([1, 0])

B @ b

# Example of A[0]*a[0] + A[1]*a[1] + A[2]*a[2] ... where a is a vector of parameters and the first is always 1

a = np.array([0, 1, 0],).T
A = A.transpose(1, 2, 0)

A_trans @ a, A @ a, A[:, :, 0]


## pagemtimes() subsitution

In [ ]:
import numpy as np









# Simple MPC QP with CasADi

In [11]:
import numpy as np
import casadi as ca
import osqp

# System Matricies

A = ca.MX(np.array([[1, 1],[0, 1]]))
B = ca.MX(np.array([[0], [1]]))
C = ca.MX(np.array([1,0]))


N = 2 # Prediction horizon 


x = ca.MX.sym('x',2,N+1)
u = ca.MX.sym('u',1,N)
y = ca.MX.sym('y',1,N)

f = 0
Q = 1 
R = 0.1
P = np.eye(2)


for i in range(N):
    f +=y[:,i].T@Q@y[:,i]+u[:,i].T@R@u[:,i]

f += x[:,-1].T@P@x[:,-1]

g=ca.vertcat(x[:,0],ca.reshape(ca.vertcat(
              u, 
             (y - C.T@x[:,1:]),
             (x[:,1]-A@x[:,:-2]-B@u)),-1,1))

lbg = ca.vertcat(np.array([[10],[0]]),-np.ones((N,1)),np.zeros((N,1)),np.zeros((N*2,1)))
ubg = ca.vertcat(np.array([[10],[0]]),np.ones((N,1)),np.zeros((N,1)),np.zeros((N*2,1)))

qp = {'f':f, 'x':ca.vertcat(ca.reshape(x,-1,1),ca.reshape(u,-1,1),ca.reshape(y,-1,1)), 'g': g}

mysol = ca.qpsol('mysol', 
                 'osqp', 
                 qp, {'print_problem':True})

mysol(lbg=lbg, ubg=ubg)





H:
[[00, 00, 00, 00, 00, 00, 00, 00, 00, 00], 
 [00, 00, 00, 00, 00, 00, 00, 00, 00, 00], 
 [00, 00, 00, 00, 00, 00, 00, 00, 00, 00], 
 [00, 00, 00, 00, 00, 00, 00, 00, 00, 00], 
 [00, 00, 00, 00, 2, 0, 00, 00, 00, 00], 
 [00, 00, 00, 00, 0, 2, 00, 00, 00, 00], 
 [00, 00, 00, 00, 00, 00, 0.2, 00, 00, 00], 
 [00, 00, 00, 00, 00, 00, 00, 0.2, 00, 00], 
 [00, 00, 00, 00, 00, 00, 00, 00, 2, 00], 
 [00, 00, 00, 00, 00, 00, 00, 00, 00, 2]]
G:[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
A:
[[1, 00, 00, 00, 00, 00, 00, 00, 00, 00], 
 [00, 1, 00, 00, 00, 00, 00, 00, 00, 00], 
 [00, 00, 00, 00, 00, 00, 1, 00, 00, 00], 
 [00, 00, -1, -0, 00, 00, 00, 00, 1, 00], 
 [-1, -1, 1, 00, 00, 00, -0, 00, 00, 00], 
 [-0, -1, 00, 1, 00, 00, -1, 00, 00, 00], 
 [00, 00, 00, 00, 00, 00, 00, 1, 00, 00], 
 [00, 00, 00, 00, -1, -0, 00, 00, 00, 1], 
 [-1, -1, 1, 00, 00, 00, 00, -0, 00, 00], 
 [-0, -1, 00, 1, 00, 00, 00, -1, 00, 00]]
lba:[10, 0, -1, -1, 0, 0, 0, 0, 0, 0]
uba:[10, 0, 1, 1, 0, 0, 0, 0, 0, 0]
lbx:[-inf, -inf, -inf, 

{'f': DM(80.9391),
 'g': DM([10, -3.37313e-06, 2.94856e-16, -1.00337, -1.68657e-06, 4.39962e-17, 1.69026e-16, 0, -1.68657e-06, 1.69826e-16]),
 'lam_g': DM([-17.9932, -17.9932, 0, -17.9932, -8.99662, 3.70974e-17, -3.36543e-13, 0, -8.99662, -3.36509e-13]),
 'lam_p': DM(0x0),
 'lam_x': DM([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'x': DM([10, -3.37313e-06, 9.99999, -3.37313e-06, 0, 0, 2.94856e-16, 1.69026e-16, 8.99662, 0])}